# Notebook 04: Deep Autoencoder & Denoising Autoencoder Anomaly Detection
**Project 26 — Card Fraud Triage (GMM vs. Autoencoder) — Team T033**  
**Author:** Vani Rudra (Task P2_03)  
**Phase:** Phase 2 (Deep Learning + Hybrid)

---

### Objectives
1. **DL Data Pipeline & Anti-Leakage Verification:** Confirm PyTorch tensors, `DataLoader` (shuffle only within training), train-legit normalization, and zero fraud contamination in training or early stopping.
2. **Undercomplete & Denoising Autoencoders:** Implement symmetric AE and DAE (`src/autoencoder.py`), explaining why `latent_dim < d_in` is mandatory.
3. **Hyperparameter Search & 5-Seed Evaluation:** Select best AE and DAE on validation PR-AUC (never test), evaluate across 5 seeds (mean ± std), and export median-seed official scores.
4. **Regularization Study:** Ablate weight decay, dropout, input noise (DAE), early stopping, batch normalization, and L1 latent sparsity, inspecting train–val gaps.
5. **Diagnostics & Technical Validation:** Learning curves, error histograms, latent-space 2D scatter, linear-AE vs PCA sanity check, top-20 FP vs caught-fraud error analysis, `base` vs `engineered` comparison, timing, and `encode()` interface tests.


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import CFG
from src.data import get_features
from src.autoencoder import (
    UndercompleteAE,
    DenoisingAE,
    compute_reconstruction_mse,
    compute_per_feature_error,
    encode_dataset,
    load_autoencoder,
)
from src.metrics import pr_auc, roc_auc, recall_at_fpr, evaluate_scores
from src.utils import set_seed, load_scores

set_seed(int(CFG.get("seed", 42)))
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


PyTorch: 2.15.0 | CUDA: False


## 1. Deep Learning Data Pipeline & Leakage Invariants
- **Tensors & `DataLoader`:** `X_train_legit` (`float32`, shape `(170524, 31)`) is wrapped in a `TensorDataset` and streamed with `batch_size=512`, `shuffle=True` strictly within training epochs.
- **Normalization Source:** `StandardScaler` is fit **only** on legitimate training transactions (`Class == 0`) inside `src/data.py` so every feature has unit variance on train-legit and no single high-variance coordinate (`Amount`) dominates MSE.
- **Why No Resampling / SMOTE:** Anomaly detection models learn the manifold of *normal* spending in one-class mode (`Class == 0` only). Oversampling or SMOTE requires fraud labels and interpolates synthetic anomalies across non-convex fraud modes, causing leakage and boundary distortion.
- **Early Stopping Hygiene:** Early stopping (`patience=10`) monitors reconstruction MSE on `X_val_legit` (56,904 legitimate validation rows), never touching the 57 validation frauds or the test set.


In [2]:
splits = get_features("base")
X_train_legit = splits.X_train_legit
X_val_legit = splits.X_val[splits.y_val == 0]

assert len(X_train_legit) == int((splits.y_train_all == 0).sum()), "Leakage: fraud in train_legit!"
assert len(X_val_legit) == int((splits.y_val == 0).sum()), "Leakage: fraud in val_legit!"
assert np.all(np.isfinite(X_train_legit)) and np.all(np.isfinite(splits.X_val)) and np.all(np.isfinite(splits.X_test))

print(f"d_in={X_train_legit.shape[1]}, train_legit={X_train_legit.shape}, val={splits.X_val.shape}, test={splits.X_test.shape}")
print(f"Train tensor memory: {X_train_legit.nbytes / (1024**2):.2f} MB")


d_in=31, train_legit=(170524, 31), val=(56961, 31), test=(56962, 31)
Train tensor memory: 20.17 MB


## 2. Why the Bottleneck Must Be Undercomplete
If `latent_dim >= d_in` (overcomplete without strong structural constraints), minimizing reconstruction MSE allows the network to learn the identity mapping $g_{\phi}(f_{\theta}(\mathbf{x})) \approx \mathbf{x}$ everywhere in $\mathbb{R}^d$. Both legitimate and fraudulent transactions would achieve near-zero reconstruction error, destroying anomaly contrast. Constraining `latent_dim < d_in` forces the encoder to compress normal correlations into a compact $k$-dimensional manifold so off-manifold frauds incur elevated MSE.


## 3. Hyperparameter Search (Validation PR-AUC Selection)
30 configurations (20 Undercomplete AE, 10 Denoising AE) were evaluated across `latent_dim` $\in \{2,4,8,12,16\}$, hidden widths, `dropout` $\in \{0,0.1,0.2\}$, `weight_decay` $\in \{0,10^{-5},10^{-4}\}$, activations $\in \{\text{relu}, \text{leaky\_relu}, \text{tanh}\}$, and noise $\sigma \in \{0.05,0.1,0.2,0.5\}$. Selection uses **validation PR-AUC only** (test labels are never inspected during tuning); `val_legit` MSE is reported alongside for transparency.


In [3]:
hp_df = pd.read_csv(ROOT / "results/tables/ae_hyperparam_search.csv")
display_cols = ["config_id", "is_dae", "latent_dim", "hidden", "act", "dropout", "weight_decay", "noise_sigma", "best_val_legit_loss", "val_pr_auc", "val_recall@0.5%"]
print(hp_df[display_cols].head(15).to_string(index=False))


 config_id  is_dae  latent_dim   hidden        act  dropout  weight_decay  noise_sigma  best_val_legit_loss  val_pr_auc  val_recall@0.5%
        11   False          12 [32, 16] leaky_relu      0.0       0.00001         0.00             0.241853     0.47702          0.70175
        29    True          16 [24, 16] leaky_relu      0.0       0.00001         0.20             0.346729     0.46091          0.73684
        26    True          12 [24, 16] leaky_relu      0.0       0.00001         0.20             0.390650     0.45475          0.77193
         5   False           8 [24, 16]       relu      0.0       0.00010         0.00             0.440969     0.43887          0.73684
        15   False          12 [24, 20]       relu      0.0       0.00010         0.00             0.323296     0.39634          0.71930
        22    True           8 [24, 16]       relu      0.0       0.00001         0.10             0.431344     0.28835          0.71930
        12   False          12 [32, 16] l

## 4. Five-Seed Reproducibility & Official Scores (Median Seed)


In [4]:
ae_5seed = pd.read_csv(ROOT / "results/tables/ae_5seed_results.csv")
dae_5seed = pd.read_csv(ROOT / "results/tables/ae_dae_5seed_results.csv")
print("=== Undercomplete AE (5 seeds) ===")
print(ae_5seed.to_string(index=False))
print("\n=== Denoising AE (5 seeds) ===")
print(dae_5seed.to_string(index=False))


=== Undercomplete AE (5 seeds) ===
 Unnamed: 0     seed          val_pr_auc         test_pr_auc        val_roc_auc       test_roc_auc     val_recall@0.1%   test_recall@0.1%    val_recall@0.5% test_recall@0.5%    val_recall@1.0%   test_recall@1.0%       best_epoch       best_val_loss   train_time_sec
          0       42 0.47702410738508844  0.4581331941182339 0.9261134789032189 0.9137007283444958  0.5964912280701754 0.5466666666666666 0.7017543859649122             0.76 0.7192982456140351 0.7733333333333333               85 0.24185267622981751            29.04
          1      123 0.09078950781461036 0.26732554921631146 0.9221933647559077 0.9241223243740515 0.15789473684210525               0.44 0.5087719298245614             0.64 0.6491228070175439 0.7066666666666667               89  0.2552775431956564            30.15
          2      456 0.20318193214769292 0.36045461593598427 0.9035352862685322 0.9175303086235285 0.45614035087719296 0.5466666666666666 0.7368421052631579           

## 5. Official Evaluation Metrics & Feature-Set Comparison (`base` vs `engineered`)


In [5]:
ae_metrics = pd.read_csv(ROOT / "results/tables/ae_metrics.csv", index_col=0)
dae_metrics = pd.read_csv(ROOT / "results/tables/ae_dae_metrics.csv", index_col=0)
base_vs_eng = pd.read_csv(ROOT / "results/tables/ae_base_vs_engineered.csv")

print("=== AE Metrics (base) ===")
print(ae_metrics.to_string())
print("\n=== DAE Metrics (base) ===")
print(dae_metrics.to_string())
print("\n=== Base vs Engineered Feature Set ===")
print(base_vs_eng.to_string(index=False))


=== AE Metrics (base) ===
                              val          test
PR-AUC                   0.227154      0.357374
ROC-AUC                  0.913885      0.902147
base_rate                0.001001      0.001317
n_fraud                 57.000000     75.000000
n_total              56961.000000  56962.000000
recall@FPR=0.001         0.368421      0.533333
F1@FPR=0.001             0.311111      0.503401
precision@FPR=0.001      0.269231      0.513889
recall@FPR=0.005         0.666667      0.680000
F1@FPR=0.005             0.200000      0.260417
precision@FPR=0.005      0.117647      0.161812
recall@FPR=0.010         0.701754      0.800000
F1@FPR=0.010             0.119940      0.193656
precision@FPR=0.010      0.065574      0.110687
precision@cap=0.001      0.333333      0.561404
precision@cap=0.005      0.133333      0.175439
precision@cap=0.010      0.070175      0.101754
precision@K=50           0.340000      0.580000
recall@K=50              0.298246      0.386667
precision@K=10

## 6. Regularization Study & Over/Under-Fit Evidence


In [6]:
reg_df = pd.read_csv(ROOT / "results/tables/ae_regularization_study.csv")
print(reg_df.to_string(index=False))
display(Image(filename=str(ROOT / "results/figures/ae_regularization_gap.png")))


 Unnamed: 0                  config  val_pr_auc  val_roc_auc  val_recall@0.1%FPR  val_recall@0.5%FPR  train_legit_mse  val_legit_mse  train_val_gap  best_epoch  epochs_trained  n_params
          0       no_regularization     0.29431      0.94364             0.45614             0.71930         0.281426       0.428811       0.147384          96             100      2631
          1           +weight_decay     0.14821      0.94700             0.29825             0.66667         0.277857       0.436539       0.158682          96             100      2631
          2                +dropout     0.04623      0.95689             0.00000             0.42105         0.687000       0.914348       0.227348          97             100      2631
          3 +input_noise_DAE(σ=0.1)     0.13643      0.95491             0.35088             0.61404         0.270579       0.432314       0.161736          90             100      2631
          4   +early_stopping(p=10)     0.29431      0.94364          

## 7. Learning Curves, Error Histograms & Latent Space Visualization


In [7]:
for fig_name in [
    "ae_learning_curves.png",
    "ae_dae_learning_curves.png",
    "ae_error_histogram.png",
    "ae_dae_error_histogram.png",
    "ae_latent_dim2_scatter.png",
    "ae_latent_scatter.png",
]:
    p = ROOT / "results/figures" / fig_name
    if p.exists():
        print(fig_name)
        display(Image(filename=str(p)))


ae_learning_curves.png
[Image: ae_learning_curves.png]
ae_dae_learning_curves.png
[Image: ae_dae_learning_curves.png]
ae_error_histogram.png
[Image: ae_error_histogram.png]
ae_dae_error_histogram.png
[Image: ae_dae_error_histogram.png]
ae_latent_dim2_scatter.png
[Image: ae_latent_dim2_scatter.png]
ae_latent_scatter.png
[Image: ae_latent_scatter.png]


## 8. Linear-AE ≈ PCA Sanity Check & Per-Feature Error Diagnostics


In [8]:
pca_sanity = pd.read_csv(ROOT / "results/tables/ae_pca_sanity_check.csv")
print("=== Linear AE vs PCA Sanity Check ===")
print(pca_sanity.to_string(index=False))

top20_dom = pd.read_csv(ROOT / "results/tables/ae_top20_dominating_features.csv")
print("\n=== Top-20 FP vs Top-20 Caught Frauds — Dominating Features (Top 10) ===")
print(top20_dom.head(10).to_string(index=False))

top20_detail = pd.read_csv(ROOT / "results/tables/ae_top20_fp_vs_tp.csv")
print("\n=== Sample Top-5 FP & Top-5 Caught Frauds ===")
print(top20_detail.groupby("group").head(5).to_string(index=False))

display(Image(filename=str(ROOT / "results/figures/ae_per_feature_error.png")))


=== Linear AE vs PCA Sanity Check ===
 Unnamed: 0  k  pca_val_legit_mse  linear_ae_val_legit_mse  mse_abs_diff  pca_val_pr_auc  linear_ae_val_pr_auc  score_pearson_corr
          0  8           0.912207                 0.903118      0.009089         0.03311               0.03093             0.99334
          1 16           0.605961                 0.600552      0.005409         0.07049               0.03970             0.93399

=== Top-20 FP vs Top-20 Caught Frauds — Dominating Features (Top 10) ===
 Unnamed: 0 feature  top20_fp_mean_sqerr  top20_tp_mean_sqerr  tp_to_fp_ratio
         13     V14            17.769419            23.858297        1.342661
         16     V17             6.856515            18.703398        2.727829
          3      V4             1.263777             9.815179        7.766543
         26     V27            11.644637             9.180696        0.788405
         22     V23             3.552176             7.347532        2.068460
         17     V18        

## 9. Encoder `encode()` Interface Verification & Technical Validation


In [9]:
tech_val = pd.read_csv(ROOT / "results/tables/ae_technical_validation.csv")
print("=== Technical Validation & Timing ===")
print(tech_val.to_string(index=False))

# Model saving / loading & encode() contract verification
ae_loaded = load_autoencoder(ROOT / "results/models/ae_best.pt", ROOT / "results/models/ae_config.json")
z_val = encode_dataset(ae_loaded, splits.X_val)
assert z_val.shape == (len(splits.X_val), ae_loaded.latent_dim), f"Unexpected shape {z_val.shape}"
assert z_val.dtype == np.float32, f"Unexpected dtype {z_val.dtype}"
assert np.all(np.isfinite(z_val)), "Non-finite latent codes!"

# Verify saved score files match loaded model MSE and satisfy score contract
ae_val_saved = load_scores("ae", "val")
ae_test_saved = load_scores("ae", "test")
dae_val_saved = load_scores("ae_dae", "val")
dae_test_saved = load_scores("ae_dae", "test")

ae_val_recomputed = compute_reconstruction_mse(ae_loaded, splits.X_val)
np.testing.assert_allclose(ae_val_saved, ae_val_recomputed, rtol=1e-5, atol=1e-5)

for name, s, y in [
    ("ae_val", ae_val_saved, splits.y_val),
    ("ae_test", ae_test_saved, splits.y_test),
    ("ae_dae_val", dae_val_saved, splits.y_val),
    ("ae_dae_test", dae_test_saved, splits.y_test),
]:
    assert s.ndim == 1 and len(s) == len(y)
    assert np.all(np.isfinite(s))
    assert s[y == 1].mean() > s[y == 0].mean(), f"{name}: fraud mean not > legit mean!"
    assert pr_auc(y, s) > y.mean(), f"{name}: PR-AUC does not beat base rate!"

print("ALL INTERFACE, SAVE/LOAD, LEAKAGE, AND SCORE-CONTRACT CHECKS PASSED!")
print(f"encode() output shape={z_val.shape}, dtype={z_val.dtype}")


=== Technical Validation & Timing ===
 Unnamed: 0           model  n_params  nan_detected  grad_norm_min  grad_norm_max  grad_norm_final  heldout_legit_batch_mse  heldout_fraud_batch_mse  fraud_to_legit_mse_ratio  avg_epoch_sec  total_train_sec  inference_10k_sec
          0 UndercompleteAE      3531         False         0.2325         0.8147           0.4913                  0.22246                  2.44451                     10.99         0.2987            29.87             0.0018
          1     DenoisingAE      2895         False         0.2110         0.6871           0.5654                  0.31821                  4.96121                     15.59         0.3438            34.38             0.0014
ALL INTERFACE, SAVE/LOAD, LEAKAGE, AND SCORE-CONTRACT CHECKS PASSED!
encode() output shape=(56961, 12), dtype=float32
